<div style="background-color:#002147; color:white; padding:10px; border-radius:5px; font-size:18px"> 1. Aspectos generales — Modelo 2: LRFM (Length, Recency, Frequency, Monetary)

<div style="background-color:#9B9B9B; color:white; padding:10px; border-radius:5px; font-size:13px">

Responsable: **Sebastian**.

Extiende el modelo RFM incorporando la dimensión **Length (L = antigüedad entre primera y última compra)**, calculado 100 % en Apache Spark nativo desde `../silver/merge_pyspark`.

Flujo: `../silver/merge_pyspark` ---> métricas L, R, F, M ---> quintiles `percent_rank` ---> segmentos LRFM ---> `../gold/hym_lrfm`

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window
import matplotlib.pyplot as plt

In [ ]:
spark = (
    SparkSession.builder
    .appName("HYM_Lab13_02_LRFM")
    .config("spark.sql.adaptive.enabled", "true")
    .config("spark.sql.adaptive.coalescePartitions.enabled", "true")
    .config("spark.driver.memory", "4g")
    .config("spark.executor.memory", "4g")
    .config("spark.sql.shuffle.partitions", "50")
    .config("spark.memory.fraction", "0.8")
    .getOrCreate()
)

spark.sparkContext.setCheckpointDir("/tmp/spark_checkpoint")
spark

<div style="background-color:#002147; color:white; padding:10px; border-radius:5px; font-size:18px"> 2. Rutas de capas

In [ ]:
SILVER = "../silver"
GOLD   = "../gold"

import os
os.makedirs(GOLD, exist_ok=True)

<div style="background-color:#002147; color:white; padding:10px; border-radius:5px; font-size:18px"> 3. Carga desde capa Silver y fecha de referencia

In [ ]:
%%time
df_silver = spark.read.parquet(f"{SILVER}/merge_pyspark")
df_clean = (
    df_silver
    .select("customer_id", "Fecha", "price")
    .filter(F.col("price").isNotNull() & (F.col("price") > 0))
)
df_clean.createOrReplaceTempView("clean_transactions")
fecha_max = df_clean.agg(F.max("Fecha")).collect()[0][0]
print(f"Registros en Silver: {df_clean.count():,} | Fecha máxima del dataset: {fecha_max}")

<div style="background-color:#002147; color:white; padding:10px; border-radius:5px; font-size:18px"> 4. Cálculo nativo de L, R, F, M en Spark SQL

<div style="background-color:#9B9B9B; color:white; padding:10px; border-radius:5px; font-size:13px">

- **L (Length):** `DATEDIFF(MAX(Fecha), MIN(Fecha))` días de relación del cliente.
- **R (Recency):** `DATEDIFF(fecha_corte, MAX(Fecha))` días desde la última compra (menor = mejor).
- **F (Frequency):** `COUNT(1)` transacciones.
- **M (Monetary):** `SUM(price)` gasto acumulado.

In [ ]:
%%time
df_lrfm_raw = spark.sql("""
WITH max_dt AS (
    SELECT MAX(Fecha) AS fecha_corte FROM clean_transactions
)
SELECT
    t.customer_id,
    CAST(DATEDIFF(MAX(t.Fecha), MIN(t.Fecha)) AS INT) AS Length,
    CAST(DATEDIFF(m.fecha_corte, MAX(t.Fecha)) AS INT) AS Recency,
    CAST(COUNT(1) AS INT) AS Frequency,
    CAST(SUM(t.price) AS DOUBLE) AS Monetary
FROM clean_transactions t
CROSS JOIN max_dt m
GROUP BY t.customer_id, m.fecha_corte
""")
df_lrfm_raw = df_lrfm_raw.checkpoint()
print(f"Clientes LRFM: {df_lrfm_raw.count():,}")
df_lrfm_raw.select("Length", "Recency", "Frequency", "Monetary").describe().show()

<div style="background-color:#002147; color:white; padding:10px; border-radius:5px; font-size:18px"> 5. Scoring por quintiles (percent_rank) y segmentos de negocio

In [ ]:
def quintil(col_pct, invertido=False):
    if invertido:
        return (
            F.when(col_pct <= 0.2, 5)
             .when(col_pct <= 0.4, 4)
             .when(col_pct <= 0.6, 3)
             .when(col_pct <= 0.8, 2)
             .otherwise(1)
        )
    return (
        F.when(col_pct <= 0.2, 1)
         .when(col_pct <= 0.4, 2)
         .when(col_pct <= 0.6, 3)
         .when(col_pct <= 0.8, 4)
         .otherwise(5)
    )

df_scored = (
    df_lrfm_raw
    .withColumn("l_pct", F.percent_rank().over(Window.orderBy(F.col("Length").asc())))
    .withColumn("r_pct", F.percent_rank().over(Window.orderBy(F.col("Recency").asc())))
    .withColumn("f_pct", F.percent_rank().over(Window.orderBy(F.col("Frequency").asc())))
    .withColumn("m_pct", F.percent_rank().over(Window.orderBy(F.col("Monetary").asc())))
    .withColumn("L_score", quintil(F.col("l_pct"), invertido=False))
    .withColumn("R_score", quintil(F.col("r_pct"), invertido=True))
    .withColumn("F_score", quintil(F.col("f_pct"), invertido=False))
    .withColumn("M_score", quintil(F.col("m_pct"), invertido=False))
    .withColumn("LRFM_score", F.round((F.col("L_score") + F.col("R_score") + F.col("F_score") + F.col("M_score")) / 4.0, 2))
    .withColumn(
        "segmento_lrfm",
        F.when((F.col("L_score") >= 4) & (F.col("R_score") >= 4) & (F.col("F_score") >= 4), "Campeones_Veteranos")
         .when((F.col("L_score") <= 2) & (F.col("R_score") >= 4) & (F.col("M_score") >= 4), "Nuevos_Alto_Valor")
         .when((F.col("L_score") >= 4) & (F.col("R_score") <= 2) & (F.col("F_score") >= 3), "Veteranos_En_Riesgo")
         .when((F.col("R_score") >= 3) & (F.col("F_score") >= 3), "Clientes_Estables")
         .otherwise("Ocasionales_Hibernando"),
    )
).checkpoint()

perfil_lrfm = (
    df_scored.groupBy("segmento_lrfm")
    .agg(
        F.count("*").alias("clientes"),
        F.round(F.avg("Length"), 1).alias("L_dias_prom"),
        F.round(F.avg("Recency"), 1).alias("R_dias_prom"),
        F.round(F.avg("Frequency"), 1).alias("F_prom"),
        F.round(F.avg("Monetary"), 3).alias("M_prom"),
    )
    .orderBy(F.col("clientes").desc())
)
perfil_lrfm.show(truncate=False)

<div style="background-color:#002147; color:white; padding:10px; border-radius:5px; font-size:18px"> 6. Gráfico resumen y persistencia en Gold

In [ ]:
pdf_perf = perfil_lrfm.toPandas()
fig, ax = plt.subplots(figsize=(9, 4))
ax.barh(pdf_perf["segmento_lrfm"], pdf_perf["clientes"], color="#002147")
ax.set_title("Distribución de Clientes por Segmento LRFM")
ax.set_xlabel("Cantidad de Clientes")
plt.tight_layout()
plt.show()

df_scored.write.mode("overwrite").parquet(f"{GOLD}/hym_lrfm")
print(f"Modelo LRFM guardado en {GOLD}/hym_lrfm | filas={spark.read.parquet(f'{GOLD}/hym_lrfm').count():,}")

In [ ]:
spark.stop()